<a href="https://colab.research.google.com/github/LulutheBA/ANLOK-WATER-PREDICTION-PROJECT-/blob/main/Copy_of_notebook_1%262.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Required Packages
Install any needed Python packages.

In [ ]:
!pip install pandas numpy matplotlib plotly scikit-learn xgboost shap >/dev/null

Mount Google Drive
Mount the user’s Google Drive to access files.


In [ ]:
from google.colab import drive
drive.mount('/content/drive') # will prompt for authentication【1†L1012-L1015】

Mounted at /content/drive


GitHub Authentication
Securely enter a GitHub token to clone the private repository.


In [ ]:

from getpass import getpass
# Prompt for a GitHub personal access token (do NOT share your token publicly)
token = getpass("Enter your GitHub Personal Access Token: ")
# Define Git repo URL (replace <username> and <repo> with actual values)
github_user = "LulutheBA" # <-- placeholder
github_repo = "ANLOK-WATER-PREDICTION-PROJECT-" # <-- placeholder (repo name)
!rm -rf $github_repo # remove any existing folder
!git clone https://{token}@github.com/{github_user}/{github_repo}.git

Enter your GitHub Personal Access Token: ··········
Cloning into 'ANLOK-WATER-PREDICTION-PROJECT-'...
remote: Enumerating objects: 382, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 382 (delta 23), reused 2 (delta 2), pack-reused 302 (from 2)
Receiving objects: 100% (382/382), 764.07 KiB | 2.51 MiB/s, done.
Resolving deltas: 100% (171/171), done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


5: Verify Repository Files
List the cloned repository contents to confirm cloning.


In [ ]:

import os
repo_path = f"/content/{github_repo}"
print("Repository structure:")
!ls "$repo_path"

Repository structure:
Data  notebook_1.ipynb	Notebooks  Outputs  Training


Import Libraries
Import all necessary Python libraries.


In [ ]:
# Data manipulation
import pandas as pd
import numpy as np
# Visualization
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
# Statistics
from scipy.stats import zscore
# Utilities
import glob
import os
import warnings
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 6)

7: Locate All CSV Files
Automatically find all .csv files in the repository.


In [ ]:
# Use glob to recursively find CSV files (pandas, glob can locate files)
csv_files = glob.glob(os.path.join(repo_path, "**/*.csv"), recursive=True)
print(f"Found {len(csv_files)} CSV files.")
for file in csv_files:
    print(file)

Found 28 CSV files.
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Verified - River Flow ( National ).csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/NIWIS_WasteWaterQuality_16-Jul-2026 .csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Potable Systems_ 2026_04_21.csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Surface Water Storage (2014-2020).csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Weather4355517.csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/TimeSeries Population_ 2026_04_21.csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Vulnerability Score Per leg_ 2026_04_21.csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Operational_ 2026_04_21.csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Piped Water Households _ 2026_04_21.csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/NIWIS_Water Supply Reliability - population_16-Jul-2026 .csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Weather.csv
/content/ANLOK-WATER-PREDICTION-PROJEC

8: Load Datasets into Dictionary
Load each CSV into a pandas DataFrame stored in a dictionary.


In [ ]:
datasets = {}
for file in csv_files:
    name = os.path.splitext(os.path.basename(file))[0]
    try:
        datasets[name] = pd.read_csv(file)
    except Exception as e:
        print(f"Skipping {name}: {e}")
        continue
print("Datasets loaded:", list(datasets.keys()))

Skipping Weather4355517: No columns to parse from file
Datasets loaded: ['Verified - River Flow ( National )', 'NIWIS_WasteWaterQuality_16-Jul-2026 ', 'Potable Systems_ 2026_04_21', 'Surface Water Storage (2014-2020)', 'TimeSeries Population_ 2026_04_21', 'Vulnerability Score Per leg_ 2026_04_21', 'Operational_ 2026_04_21', 'Piped Water Households _ 2026_04_21', 'NIWIS_Water Supply Reliability - population_16-Jul-2026 ', 'Weather', 'Timeseries Households_ 2026_04_21', 'NIWIS_RWT_16-Jul-2026 ', 'Protests Identified by Municipal IQ_ 2026_04_21', 'People Served with Water_ 2026_04_21', 'NIWIS_Access to Water Infrastructure Delivered- Population_16-Jul-2026 ', 'Areas of Highest Vulnerability_ 2026_04_21', 'Water demand growth_ 2026_04_21', 'Total Population_ 2026_04_21', 'NIWIS_GroundwaterStatus_16-Jul-2026 ', 'Residential Tariffs_ 2026_04_21', 'Households Served with Water_ 2026_04_21', 'Drinking Water Quality Compliance - SANS 241 _2006 - National [From_ 2022-07-01 To_ 2026-06-30]', 'Ins

9: Create Inventory Table
Compute inventory (rows, columns, missing, duplicates, etc.) and display as DataFrame.

In [ ]:
inventory = []
for name, df in datasets.items():
    inventory.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Memory(MB)": round(df.memory_usage().sum() / 1024**2, 2),
        "Missing Values": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum(),
        "Numeric Columns": len(df.select_dtypes(include=np.number).columns),
        "Categorical Columns": len(df.select_dtypes(exclude=np.number).columns)
    })
inventory_df = pd.DataFrame(inventory)
display(inventory_df)

,Dataset,Rows,Columns,Memory(MB),Missing Values,Duplicate Rows,Numeric Columns,Categorical Columns
0,Verified - River Flow ( National ),7,2,0.00,0,0,1,1
1,NIWIS_WasteWaterQuality_16-Jul-2026,143,6,0.01,143,0,1,5
2,Potable Systems_ 2026_04_21,61,11,0.01,69,0,2,9
3,Surface Water Storage (2014-2020),374,9,0.03,1971,0,8,1
4,TimeSeries Population_ 2026_04_21,11,35,0.00,43,0,0,35
5,Vulnerability Score Per leg_ 2026_04_21,11,9,0.00,17,0,6,3
6,Operational_ 2026_04_21,11,8,0.00,16,0,2,6
7,Piped Water Households _ 2026_04_21,10,12,0.00,19,0,2,10
8,NIWIS_Water Supply Reliability - population_16...,144,14,0.02,189,0,1,13
9,Weather,21301,14,2.28,38846,0,7,7


10: Save Inventory Table
Save the inventory table to CSV for record.



In [ ]:
outputs_dir = "Outputs"
os.makedirs(outputs_dir, exist_ok=True)
inventory_df.to_csv(os.path.join(outputs_dir, "dataset_inventory.csv"),
index=False)
print("Inventory saved to Outputs/dataset_inventory.csv")

Inventory saved to Outputs/dataset_inventory.csv


11: Preview One Dataset
Example: show first rows of one dataset to verify loading.

In [ ]:
# Replace with a valid dataset key if needed; here we try the first one
sample_key = list(datasets.keys())[0]
print(f"Sample dataset ({sample_key}):")
display(datasets[sample_key].head())

Sample dataset (Verified - River Flow ( National )):


,'Flow,'Flow Level :
0,'High,243
1,'Moderately High,54
2,'Normal,66
3,'Moderately Low,36
4,'Low,46


In [ ]:
# ============================================
# CELL 1: IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np
import os
import re
import glob
import warnings

from IPython.display import display

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# ============================================
# CELL 2: CONFIRM DATASETS
# ============================================

print("=" * 80)
print("DATASETS LOADED")
print("=" * 80)

print(f"Total number of datasets: {len(datasets)}")
print()

for i, name in enumerate(datasets.keys(), start=1):
    print(f"{i}. {name}")

DATASETS LOADED
Total number of datasets: 27

1. Verified - River Flow ( National )
2. NIWIS_WasteWaterQuality_16-Jul-2026 
3. Potable Systems_ 2026_04_21
4. Surface Water Storage (2014-2020)
5. TimeSeries Population_ 2026_04_21
6. Vulnerability Score Per leg_ 2026_04_21
7. Operational_ 2026_04_21
8. Piped Water Households _ 2026_04_21
9. NIWIS_Water Supply Reliability - population_16-Jul-2026 
10. Weather
11. Timeseries Households_ 2026_04_21
12. NIWIS_RWT_16-Jul-2026 
13. Protests Identified by Municipal IQ_ 2026_04_21
14. People Served with Water_ 2026_04_21
15. NIWIS_Access to Water Infrastructure Delivered- Population_16-Jul-2026 
16. Areas of Highest Vulnerability_ 2026_04_21
17. Water demand growth_ 2026_04_21
18. Total Population_ 2026_04_21
19. NIWIS_GroundwaterStatus_16-Jul-2026 
20. Residential Tariffs_ 2026_04_21
21. Households Served with Water_ 2026_04_21
22. Drinking Water Quality Compliance - SANS 241 _2006 - National [From_ 2022-07-01 To_ 2026-06-30]
23. Institutional 

In [ ]:
# ============================================
# CELL 3: COMPLETE DATASET OVERVIEW
# ============================================

overview = []

for name, df in datasets.items():

    overview.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": int(df.isnull().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Numeric Columns": len(
            df.select_dtypes(include=np.number).columns
        ),
        "Categorical Columns": len(
            df.select_dtypes(include="object").columns
        ),
        "Memory (MB)": round(
            df.memory_usage(deep=True).sum() / 1024**2,
            2
        )
    })

overview_df = pd.DataFrame(overview)

display(overview_df)

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Numeric Columns,Categorical Columns,Memory (MB)
0,Verified - River Flow ( National ),7,2,0,0,1,1,0.00
1,NIWIS_WasteWaterQuality_16-Jul-2026,143,6,143,0,1,5,0.04
2,Potable Systems_ 2026_04_21,61,11,69,0,2,9,0.04
3,Surface Water Storage (2014-2020),374,9,1971,0,8,1,0.05
4,TimeSeries Population_ 2026_04_21,11,35,43,0,0,35,0.03
5,Vulnerability Score Per leg_ 2026_04_21,11,9,17,0,6,3,0.00
6,Operational_ 2026_04_21,11,8,16,0,2,6,0.00
7,Piped Water Households _ 2026_04_21,10,12,19,0,2,10,0.01
8,NIWIS_Water Supply Reliability - population_16...,144,14,189,0,1,13,0.10
9,Weather,21301,14,38846,0,7,7,8.86


In [ ]:
# ============================================
# CELL 4: INSPECT EVERY DATASET
# ============================================

for name, df in datasets.items():

    print("\n")
    print("=" * 100)
    print(f"DATASET: {name}")
    print("=" * 100)

    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    print("\nColumn names:")
    print(list(df.columns))

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) == 0:
        print("No missing values.")
    else:
        print(missing)

    print("\nDuplicate rows:")
    print(df.duplicated().sum())

    print("\nFirst 5 rows:")
    display(df.head())

    print("\nNumerical summary:")
    display(df.describe(include="all").T)



DATASET: Verified - River Flow ( National )
Rows: 7
Columns: 2

Column names:
["'Flow", "'Flow Level :"]

Data types:
'Flow            object
'Flow Level :     int64
dtype: object

Missing values:
No missing values.

Duplicate rows:
0

First 5 rows:


,'Flow,'Flow Level :
0,'High,243
1,'Moderately High,54
2,'Normal,66
3,'Moderately Low,36
4,'Low,46



Numerical summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
'Flow,7,7,'High,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Flow Level :,7.0,NaN,NaN,NaN,76.857143,74.44557,30.0,41.0,54.0,64.5,243.0




DATASET: NIWIS_WasteWaterQuality_16-Jul-2026 
Rows: 143
Columns: 6

Column names:
['Unnamed: 0', 'WSA', 'Chemical', 'Microbiological', 'Physical', 'Monitoring']

Data types:
Unnamed: 0         float64
WSA                 object
Chemical            object
Microbiological     object
Physical            object
Monitoring          object
dtype: object

Missing values:
Unnamed: 0    143
dtype: int64

Duplicate rows:
0

First 5 rows:


...

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Region,11,11,City of Johannesburg Metropolitan Municipality,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time Frame,10,1,2 026,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unnamed: 2,1,1,"<b>Data Source:</b> <a style=""font-weight: bol...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Risk Type,10,1,Microbiological : Acute Health,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Analysis Done,10,9,0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Analysis Failed,10.0,NaN,NaN,NaN,21.6,35.450121,0.0,0.25,5.5,21.25,108.0
Compliance Percentage,10.0,NaN,NaN,NaN,76.83,40.751744,0.0,86.5275,97.08,98.895,100.0
Compliance Result,10,4,Good,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN




DATASET: Source of Water Households _ 2026_04_21
Rows: 10
Columns: 13

Column names:
['Region', 'Time Frame', 'Unnamed: 2', 'Regional/local water scheme (operated by municipality or other w', 'Borehole', 'Spring', 'Rain water tank', 'Dam/pool/stagnant water', 'River/stream', 'Water vendor', 'Water tanker', 'Other', 'Not applicable']

Data types:
Region                                                               object
Time Frame                                                           object
Unnamed: 2                                                           object
Regional/local water scheme (operated by municipality or other w     object
Borehole                                                             object
Spring                                                               object
Rain water tank                                                      object
Dam/pool/stagnant water                                              object
River/stream                              

,Region,Time Frame,Unnamed: 2,Regional/local water scheme (operated by municipality or other w,Borehole,Spring,Rain water tank,Dam/pool/stagnant water,River/stream,Water vendor,Water tanker,Other,Not applicable
0,City of Johannesburg Metropolitan Municipality,Oct 2011,NaN,1 371 966,14 478,1 215,1 556,1 531,424,5 592,20 887,15 833,0.0
1,City of Tshwane Metropolitan Municipality,Oct 2011,NaN,825 746,25 730,1 231,1 426,2 409,550,5 615,30 864,16 983,0.0
2,Ekurhuleni Metropolitan Municipality,Oct 2011,NaN,972 647,10 997,985,1 065,923,253,3 703,9 247,14 629,0.0
3,Emfuleni,Oct 2011,NaN,211 967,3 781,98,395,225,90,619,425,2 295,0.0
4,Lesedi,Oct 2011,NaN,26 377,2 495,68,55,98,28,40,193,279,0.0



Numerical summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Region,10,10,City of Johannesburg Metropolitan Municipality,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time Frame,9,1,Oct 2011,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unnamed: 2,1,1,"<b>Data Source:</b> <a style=""font-weight: bol...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Regional/local water scheme (operated by municipality or other w,9,9,1 371 966,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Borehole,9,9,14 478,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Spring,9,9,1 215,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Rain water tank,9,9,1 556,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dam/pool/stagnant water,9,9,1 531,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
River/stream,9,9,424,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Water vendor,9,9,5 592,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ============================================
# CELL 5: COLUMN-LEVEL DATA AUDIT
# ============================================

column_audit = []

for name, df in datasets.items():

    for col in df.columns:

        column_audit.append({
            "Dataset": name,
            "Column": col,
            "Data Type": str(df[col].dtype),
            "Rows": len(df),
            "Missing": int(df[col].isnull().sum()),
            "Missing %": round(
                df[col].isnull().mean() * 100,
                2
            ),
            "Unique Values": int(df[col].nunique(dropna=True)),
            "Duplicate Values": int(
                df[col].duplicated().sum()
            )
        })

column_audit_df = pd.DataFrame(column_audit)

display(column_audit_df)

,Dataset,Column,Data Type,Rows,Missing,Missing %,Unique Values,Duplicate Values
0,Verified - River Flow ( National ),'Flow,object,7,0,0.0,7,0
1,Verified - River Flow ( National ),'Flow Level :,int64,7,0,0.0,7,0
2,NIWIS_WasteWaterQuality_16-Jul-2026,Unnamed: 0,float64,143,143,100.0,0,142
3,NIWIS_WasteWaterQuality_16-Jul-2026,WSA,object,143,0,0.0,143,0
4,NIWIS_WasteWaterQuality_16-Jul-2026,Chemical,object,143,0,0.0,62,81
...,...,...,...,...,...,...,...,...
308,Source of Water Households _ 2026_04_21,River/stream,object,10,1,10.0,9,0
309,Source of Water Households _ 2026_04_21,Water vendor,object,10,1,10.0,9,0
310,Source of Water Households _ 2026_04_21,Water tanker,object,10,1,10.0,9,0
311,Source of Water Households _ 2026_04_21,Other,object,10,1,10.0,9,0


In [ ]:
# ============================================
# CELL 6: MISSING VALUE ANALYSIS
# ============================================

missing_columns = column_audit_df[
    column_audit_df["Missing"] > 0
].copy()

print(
    f"Number of columns containing missing values: "
    f"{len(missing_columns)}"
)

display(missing_columns)

Number of columns containing missing values: 259


,Dataset,Column,Data Type,Rows,Missing,Missing %,Unique Values,Duplicate Values
2,NIWIS_WasteWaterQuality_16-Jul-2026,Unnamed: 0,float64,143,143,100.00,0,142
9,Potable Systems_ 2026_04_21,Time Frame,object,61,1,1.64,1,59
10,Potable Systems_ 2026_04_21,Unnamed: 2,object,61,60,98.36,1,59
11,Potable Systems_ 2026_04_21,System name,object,61,1,1.64,30,30
12,Potable Systems_ 2026_04_21,System area,object,61,1,1.64,30,30
...,...,...,...,...,...,...,...,...
308,Source of Water Households _ 2026_04_21,River/stream,object,10,1,10.00,9,0
309,Source of Water Households _ 2026_04_21,Water vendor,object,10,1,10.00,9,0
310,Source of Water Households _ 2026_04_21,Water tanker,object,10,1,10.00,9,0
311,Source of Water Households _ 2026_04_21,Other,object,10,1,10.00,9,0


In [ ]:
# ============================================
# CELL 7: MISSING VALUES BY DATASET
# ============================================

missing_summary = (
    overview_df[
        ["Dataset", "Rows", "Columns", "Missing Values"]
    ]
    .sort_values(
        by="Missing Values",
        ascending=False
    )
)

display(missing_summary)

,Dataset,Rows,Columns,Missing Values
9,Weather,21301,14,38846
18,NIWIS_GroundwaterStatus_16-Jul-2026,1947,8,2511
3,Surface Water Storage (2014-2020),374,9,1971
23,Surface Water Storage ( 2020 -2026),375,9,1953
8,NIWIS_Water Supply Reliability - population_16...,144,14,189
1,NIWIS_WasteWaterQuality_16-Jul-2026,143,6,143
2,Potable Systems_ 2026_04_21,61,11,69
10,Timeseries Households_ 2026_04_21,11,35,43
4,TimeSeries Population_ 2026_04_21,11,35,43
15,Areas of Highest Vulnerability_ 2026_04_21,11,21,29


In [ ]:
# ============================================
# CELL 8: DUPLICATE ANALYSIS
# ============================================

duplicate_summary = (
    overview_df[
        ["Dataset", "Rows", "Duplicate Rows"]
    ]
    .sort_values(
        by="Duplicate Rows",
        ascending=False
    )
)

display(duplicate_summary)

,Dataset,Rows,Duplicate Rows
18,NIWIS_GroundwaterStatus_16-Jul-2026,1947,1
1,NIWIS_WasteWaterQuality_16-Jul-2026,143,0
0,Verified - River Flow ( National ),7,0
3,Surface Water Storage (2014-2020),374,0
4,TimeSeries Population_ 2026_04_21,11,0
5,Vulnerability Score Per leg_ 2026_04_21,11,0
6,Operational_ 2026_04_21,11,0
7,Piped Water Households _ 2026_04_21,10,0
8,NIWIS_Water Supply Reliability - population_16...,144,0
9,Weather,21301,0


In [ ]:
# ============================================
# CELL 9: DISPLAY DUPLICATE ROWS
# ============================================

for name, df in datasets.items():

    duplicate_rows = df[df.duplicated(keep=False)]

    if not duplicate_rows.empty:

        print("\n")
        print("=" * 100)
        print(f"DUPLICATES FOUND: {name}")
        print("=" * 100)

        print(
            f"Number of duplicate rows: "
            f"{len(duplicate_rows)}"
        )

        display(duplicate_rows.head(20))



DUPLICATES FOUND: NIWIS_GroundwaterStatus_16-Jul-2026 
Number of duplicate rows: 2


,Quaternary,Reserve Study,Available (GRA2)(m3/a),Recharge (GWR)(m3/a),Reserve (GWR)(m3/a),Abstracted (WARMS)(m3/a),Surplus (GRA2)(m3/a),Surplus (GWR)(m3/a)
195,H70K,0,335 546,NaN,NaN,74 073,261 473,NaN
196,H70K,0,335 546,NaN,NaN,74 073,261 473,NaN


In [ ]:
# ============================================
# CELL 10: IDENTIFY DATE/YEAR COLUMNS
# ============================================

date_columns = []

for name, df in datasets.items():

    for col in df.columns:

        col_lower = str(col).lower()

        if (
            "date" in col_lower
            or "year" in col_lower
            or "time" in col_lower
            or "month" in col_lower
        ):

            date_columns.append({
                "Dataset": name,
                "Column": col,
                "Data Type": str(df[col].dtype),
                "Example Values": df[col]
                    .dropna()
                    .astype(str)
                    .head(5)
                    .tolist()
            })

date_columns_df = pd.DataFrame(date_columns)

display(date_columns_df)

,Dataset,Column,Data Type,Example Values
0,Potable Systems_ 2026_04_21,Time Frame,object,"[April 2022, April 2022, April 2022, April 202..."
1,Surface Water Storage (2014-2020),'DateTime,object,"['2014-10-01 00:00:00, '2014-10-08 00:00:00, '..."
2,TimeSeries Population_ 2026_04_21,Time Frame,object,"[April 2026, April 2026, April 2026, April 202..."
3,Vulnerability Score Per leg_ 2026_04_21,Time Frame,object,"[2 018, 2 018, 2 018, 2 018, 2 018]"
4,Operational_ 2026_04_21,Time Frame,object,"[2 026, 2 026, 2 026, 2 026, 2 026]"
5,Piped Water Households _ 2026_04_21,Time Frame,object,"[Oct 2011, Oct 2011, Oct 2011, Oct 2011, Oct 2..."
6,Weather,DATE,object,"[2000-01-01, 2000-01-02, 2000-01-03, 2000-01-0..."
7,Timeseries Households_ 2026_04_21,Time Frame,object,"[April 2026, April 2026, April 2026, April 202..."
8,NIWIS_RWT_16-Jul-2026,Date,int64,"[2025, 2025, 2025, 2025, 2025]"
9,Protests Identified by Municipal IQ_ 2026_04_21,Time Frame,object,"[2 018, 2 018, 2 018, 2 018, 2 018]"


In [ ]:
# ============================================
# CELL 11: IDENTIFY LOCATION COLUMNS
# ============================================

location_columns = []

location_keywords = [
    "region",
    "municipality",
    "province",
    "district",
    "city",
    "location",
    "area"
]

for name, df in datasets.items():

    for col in df.columns:

        col_lower = str(col).lower()

        if any(
            keyword in col_lower
            for keyword in location_keywords
        ):

            location_columns.append({
                "Dataset": name,
                "Column": col,
                "Unique Values": df[col].nunique(
                    dropna=True
                ),
                "Example Values": df[col]
                    .dropna()
                    .astype(str)
                    .unique()[:5]
                    .tolist()
            })

location_columns_df = pd.DataFrame(
    location_columns
)

display(location_columns_df)

,Dataset,Column,Unique Values,Example Values
0,Potable Systems_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
1,Potable Systems_ 2026_04_21,System area,30,"[Alexandra,East Bank ,Tsutsumani ,Bultfontein ..."
2,TimeSeries Population_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
3,Vulnerability Score Per leg_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
4,Vulnerability Score Per leg_ 2026_04_21,4# Technical Staff Capacity (Numbers),8,"[79.0, 20.0, 80.0, 15.0, 69.0]"
5,Operational_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
6,Piped Water Households _ 2026_04_21,Region,10,[City of Johannesburg Metropolitan Municipalit...
7,Timeseries Households_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
8,Protests Identified by Municipal IQ_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
9,Protests Identified by Municipal IQ_ 2026_04_21,Electricity,8,"[76.0, 29.0, 31.0, 1.0, 2.0]"


In [ ]:
# ============================================
# CELL 12: NUMERICAL VARIABLES
# ============================================

numeric_summary = []

for name, df in datasets.items():

    numeric_cols = df.select_dtypes(
        include=np.number
    ).columns

    for col in numeric_cols:

        numeric_summary.append({
            "Dataset": name,
            "Column": col,
            "Min": df[col].min(),
            "Max": df[col].max(),
            "Mean": df[col].mean(),
            "Median": df[col].median(),
            "Std": df[col].std(),
            "Missing": df[col].isnull().sum()
        })

numeric_summary_df = pd.DataFrame(
    numeric_summary
)

display(numeric_summary_df)

,Dataset,Column,Min,Max,Mean,Median,Std,Missing
0,Verified - River Flow ( National ),'Flow Level :,30.000000,243.00000,76.857143,54.000000,74.445570,0
1,NIWIS_WasteWaterQuality_16-Jul-2026,Unnamed: 0,NaN,NaN,NaN,NaN,NaN,143
2,Potable Systems_ 2026_04_21,Drinking Water Quality Percentage,0.000000,100.00000,90.268000,99.895000,26.286442,1
3,Potable Systems_ 2026_04_21,Latest Blue Drop Score,42.300000,98.10000,77.852667,86.515000,17.270366,1
4,Surface Water Storage (2014-2020),'High,90.990000,98.70000,95.099167,94.245000,2.235766,302
...,...,...,...,...,...,...,...,...
65,Surface Water Storage ( 2020 -2026),'2025/2026,60.947382,100.84599,88.218523,91.374867,8.355380,73
66,Total Households_ 2026_04_21,HH Density,2.510000,3.14000,2.861000,2.870000,0.217432,1
67,Microbiological _ Acute Health_ 2026_04_21,Analysis Failed,0.000000,108.00000,21.600000,5.500000,35.450121,1
68,Microbiological _ Acute Health_ 2026_04_21,Compliance Percentage,0.000000,100.00000,76.830000,97.080000,40.751744,1


In [ ]:
# ============================================
# CELL 13: OUTLIER AUDIT USING IQR
# ============================================

outlier_summary = []

for name, df in datasets.items():

    numeric_cols = df.select_dtypes(
        include=np.number
    ).columns

    for col in numeric_cols:

        series = df[col].dropna()

        if len(series) < 4:
            continue

        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = (
            (series < lower_bound)
            |
            (series > upper_bound)
        )

        outlier_summary.append({
            "Dataset": name,
            "Column": col,
            "Q1": Q1,
            "Q3": Q3,
            "IQR": IQR,
            "Lower Bound": lower_bound,
            "Upper Bound": upper_bound,
            "Outlier Count": int(outliers.sum()),
            "Outlier %": round(
                outliers.mean() * 100,
                2
            )
        })

outlier_df = pd.DataFrame(outlier_summary)

display(
    outlier_df.sort_values(
        "Outlier Count",
        ascending=False
    )
)

,Dataset,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count,Outlier %
24,Weather,PRCP,0.000000,4.300000,4.300000,-6.450000,10.750000,1028,12.82
26,Weather,TMAX,22.000000,28.700000,6.700000,11.950000,38.750000,49,0.25
25,Weather,TAVG,14.300000,20.900000,6.600000,4.400000,30.800000,40,0.19
63,Surface Water Storage ( 2020 -2026),'2025/2026,84.365152,94.346891,9.981739,69.392543,109.319499,14,4.64
1,Potable Systems_ 2026_04_21,Drinking Water Quality Percentage,98.390000,100.000000,1.610000,95.975000,102.415000,8,13.33
...,...,...,...,...,...,...,...,...,...
60,Surface Water Storage ( 2020 -2026),'Low,63.657500,73.497500,9.840000,48.897500,88.257500,0,0.00
62,Surface Water Storage ( 2020 -2026),'2024/2025,76.524777,93.298052,16.773275,51.364864,118.457965,0,0.00
61,Surface Water Storage ( 2020 -2026),'Very Low,58.785500,67.150000,8.364500,46.238750,79.696750,0,0.00
64,Total Households_ 2026_04_21,HH Density,2.785000,3.040000,0.255000,2.402500,3.422500,0,0.00


In [ ]:
# ============================================
# CELL 14: MASTER AUDIT REPORT
# ============================================

audit_report = overview_df.copy()

audit_report["Missing %"] = (
    audit_report["Missing Values"]
    / audit_report["Rows"]
    * 100
).round(2)

audit_report["Duplicate %"] = (
    audit_report["Duplicate Rows"]
    / audit_report["Rows"]
    * 100
).round(2)

display(audit_report)

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Numeric Columns,Categorical Columns,Memory (MB),Missing %,Duplicate %
0,Verified - River Flow ( National ),7,2,0,0,1,1,0.00,0.00,0.00
1,NIWIS_WasteWaterQuality_16-Jul-2026,143,6,143,0,1,5,0.04,100.00,0.00
2,Potable Systems_ 2026_04_21,61,11,69,0,2,9,0.04,113.11,0.00
3,Surface Water Storage (2014-2020),374,9,1971,0,8,1,0.05,527.01,0.00
4,TimeSeries Population_ 2026_04_21,11,35,43,0,0,35,0.03,390.91,0.00
5,Vulnerability Score Per leg_ 2026_04_21,11,9,17,0,6,3,0.00,154.55,0.00
6,Operational_ 2026_04_21,11,8,16,0,2,6,0.00,145.45,0.00
7,Piped Water Households _ 2026_04_21,10,12,19,0,2,10,0.01,190.00,0.00
8,NIWIS_Water Supply Reliability - population_16...,144,14,189,0,1,13,0.10,131.25,0.00
9,Weather,21301,14,38846,0,7,7,8.86,182.37,0.00


In [ ]:
# ============================================
# CELL 15: SAVE AUDIT RESULTS
# ============================================

audit_dir = "Outputs/DataAudit"

os.makedirs(
    audit_dir,
    exist_ok=True
)

overview_df.to_csv(
    os.path.join(
        audit_dir,
        "dataset_overview.csv"
    ),
    index=False
)

column_audit_df.to_csv(
    os.path.join(
        audit_dir,
        "column_audit.csv"
    ),
    index=False
)

missing_columns.to_csv(
    os.path.join(
        audit_dir,
        "missing_columns.csv"
    ),
    index=False
)

duplicate_summary.to_csv(
    os.path.join(
        audit_dir,
        "duplicate_summary.csv"
    ),
    index=False
)

numeric_summary_df.to_csv(
    os.path.join(
        audit_dir,
        "numeric_summary.csv"
    ),
    index=False
)

outlier_df.to_csv(
    os.path.join(
        audit_dir,
        "outlier_summary.csv"
    ),
    index=False
)

print("Data audit reports saved successfully.")
print(f"Location: {audit_dir}")

Data audit reports saved successfully.
Location: Outputs/DataAudit


Notebook 2

In [ ]:
import pandas as pd
import numpy as np from scipy.stats
import zscore import os, glob,
warnings warnings.filterwarnings("ignore")

SyntaxError: invalid syntax (830179280.py, line 2)

In [ ]:
csv_files = glob.glob("/content/ANLOK-WATER-PREDICTION-PROJECT-/**/*.csv", recursive=True)
datasets = {}
for file in csv_files:
  name = os.path.splitext(os.path.basename(file))[0]
  try: datasets[name] = pd.read_csv(file)
  except Exception as e: continue
  print("Datasets to clean:", list(datasets.keys()))

Datasets to clean: ['Verified - River Flow ( National )']
Datasets to clean: ['Verified - River Flow ( National )', 'NIWIS_WasteWaterQuality_16-Jul-2026 ']
Datasets to clean: ['Verified - River Flow ( National )', 'NIWIS_WasteWaterQuality_16-Jul-2026 ', 'Potable Systems_ 2026_04_21']
Datasets to clean: ['Verified - River Flow ( National )', 'NIWIS_WasteWaterQuality_16-Jul-2026 ', 'Potable Systems_ 2026_04_21', 'Surface Water Storage (2014-2020)']
Datasets to clean: ['Verified - River Flow ( National )', 'NIWIS_WasteWaterQuality_16-Jul-2026 ', 'Potable Systems_ 2026_04_21', 'Surface Water Storage (2014-2020)', 'TimeSeries Population_ 2026_04_21']
Datasets to clean: ['Verified - River Flow ( National )', 'NIWIS_WasteWaterQuality_16-Jul-2026 ', 'Potable Systems_ 2026_04_21', 'Surface Water Storage (2014-2020)', 'TimeSeries Population_ 2026_04_21', 'Vulnerability Score Per leg_ 2026_04_21']
Datasets to clean: ['Verified - River Flow ( National )', 'NIWIS_WasteWaterQuality_16-Jul-2026 ', 'P

In [ ]:
def clean_dataframe(df):
  df = df.copy()
  # Drop exact duplicate rows
  before = df.shape[0]
  df = df.drop_duplicates()
  after = df.shape[0]
  print(f"Dropped {before-after} duplicates")
  # Standardize column names: strip, lower-case, replace spaces with underscores
  df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
  # Drop completely empty rows/columns
  df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')
  # Strip whitespace from string entries
  for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()
  return df

In [ ]:
cleaned = {}
for name, df in datasets.items():
  print(f"\nCleaning {name}:")
  df_clean = clean_dataframe(df)
  cleaned[name] = df_clean


Cleaning Verified - River Flow ( National ):
Dropped 0 duplicates

Cleaning NIWIS_WasteWaterQuality_16-Jul-2026 :
Dropped 0 duplicates

Cleaning Potable Systems_ 2026_04_21:
Dropped 0 duplicates

Cleaning Surface Water Storage (2014-2020):
Dropped 0 duplicates

Cleaning TimeSeries Population_ 2026_04_21:
Dropped 0 duplicates

Cleaning Vulnerability Score Per leg_ 2026_04_21:
Dropped 0 duplicates

Cleaning Operational_ 2026_04_21:
Dropped 0 duplicates

Cleaning Piped Water Households _ 2026_04_21:
Dropped 0 duplicates

Cleaning NIWIS_Water Supply Reliability - population_16-Jul-2026 :
Dropped 0 duplicates

Cleaning Weather:
Dropped 0 duplicates

Cleaning Timeseries Households_ 2026_04_21:
Dropped 0 duplicates

Cleaning NIWIS_RWT_16-Jul-2026 :
Dropped 0 duplicates

Cleaning Protests Identified by Municipal IQ_ 2026_04_21:
Dropped 0 duplicates

Cleaning People Served with Water_ 2026_04_21:
Dropped 0 duplicates

Cleaning NIWIS_Access to Water Infrastructure Delivered- Population_16-Jul-2

In [ ]:
for name, df in cleaned.items():
  # Example: Convert date columns to datetime if any
  for col in df.columns:
    if 'date' in col or 'year' in col:
      try:
        df[col] = pd.to_datetime(df[col])
        print(f"Converted {name}.{col} to datetime")
      except:
        pass

  # Impute or flag missing numerical values
  numeric_cols = df.select_dtypes(include=np.number).columns
  if len(numeric_cols) > 0:
    imputer = SimpleImputer(strategy='mean')
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    print(f"Imputed missing numeric values in {name}")

  # For categorical: fill missing with 'Unknown'
  cat_cols = df.select_dtypes(include='object').columns
  for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

  cleaned[name] = df

Imputed missing numeric values in Verified - River Flow ( National )
Imputed missing numeric values in Potable Systems_ 2026_04_21
Converted Surface Water Storage (2014-2020).'datetime to datetime
Imputed missing numeric values in Surface Water Storage (2014-2020)
Imputed missing numeric values in Vulnerability Score Per leg_ 2026_04_21
Imputed missing numeric values in Operational_ 2026_04_21
Imputed missing numeric values in Piped Water Households _ 2026_04_21
Converted Weather.date to datetime
Imputed missing numeric values in Weather
Converted NIWIS_RWT_16-Jul-2026 .date to datetime
Imputed missing numeric values in NIWIS_RWT_16-Jul-2026 
Imputed missing numeric values in Protests Identified by Municipal IQ_ 2026_04_21
Imputed missing numeric values in People Served with Water_ 2026_04_21
Imputed missing numeric values in Water demand growth_ 2026_04_21
Imputed missing numeric values in Total Population_ 2026_04_21
Imputed missing numeric values in NIWIS_GroundwaterStatus_16-Jul-20

In [ ]:
for name, df in cleaned.items():
  numeric = df.select_dtypes(include=np.number)
  if not numeric.empty:
    z_scores = np.abs(zscore(numeric, nan_policy='omit'))
    outliers = (z_scores > 3).sum(axis=0)
    print(f"{name} outliers per column:\n{outliers}")
    # (Optional) Remove rows with any outlier in numeric columns:
    # df = df[(z_scores < 3).all(axis=1)]
    # cleaned[name] = df

Verified - River Flow ( National ) outliers per column:
[0]
Potable Systems_ 2026_04_21 outliers per column:
[4 0]
Surface Water Storage (2014-2020) outliers per column:
[18 16 12 17 17  8  0  0]
Vulnerability Score Per leg_ 2026_04_21 outliers per column:
[0 0 0 0 0 0]
Operational_ 2026_04_21 outliers per column:
[0 0]
Piped Water Households _ 2026_04_21 outliers per column:
[0 0]
Weather outliers per column:
[  0   0   0 334  33  51  46]
NIWIS_RWT_16-Jul-2026  outliers per column:
[0 0 0]
Protests Identified by Municipal IQ_ 2026_04_21 outliers per column:
[0 0 0 0 0 0 0 0 0 0 0 0 0]
People Served with Water_ 2026_04_21 outliers per column:
[0]
Water demand growth_ 2026_04_21 outliers per column:
[0 0 0]
Total Population_ 2026_04_21 outliers per column:
[0]
NIWIS_GroundwaterStatus_16-Jul-2026  outliers per column:
[0]
Households Served with Water_ 2026_04_21 outliers per column:
[0 0]
Drinking Water Quality Compliance - SANS 241 _2006 - National [From_ 2022-07-01 To_ 2026-06-30] outl

In [ ]:
report = []
for name, df in cleaned.items():
  report.append({
      "Dataset": name,
      "Rows": len(df),
      "Columns": df.shape[1],
      "Missing": df.isnull().sum().sum(),
      "Duplicates": df.duplicated().sum()
  })
report_df = pd.DataFrame(report)
display(report_df)

,Dataset,Rows,Columns,Missing,Duplicates
0,Verified - River Flow ( National ),7,2,0.0,0
1,NIWIS_WasteWaterQuality_16-Jul-2026,143,5,0.0,0
2,Potable Systems_ 2026_04_21,61,11,0.0,0
3,Surface Water Storage (2014-2020),374,9,0.0,0
4,TimeSeries Population_ 2026_04_21,11,35,0.0,0
5,Vulnerability Score Per leg_ 2026_04_21,11,9,0.0,0
6,Operational_ 2026_04_21,11,8,0.0,0
7,Piped Water Households _ 2026_04_21,10,12,0.0,0
8,NIWIS_Water Supply Reliability - population_16...,144,13,0.0,0
9,Weather,21301,14,0.0,0


In [ ]:
clean_dir = os.path.join("Outputs", "CleanData")
os.makedirs(clean_dir, exist_ok=True)

for name, df in cleaned.items():
  out_path = os.path.join(clean_dir, f"{name}_clean.csv")
  df.to_csv(out_path, index=False)
  print(f"Saved cleaned {name} ({df.shape}) to {out_path})")

# Each cleaned CSV is saved for future use (e.g. merged dataset)

Saved cleaned Verified - River Flow ( National ) ((7, 2)) to Outputs/CleanData/Verified - River Flow ( National )_clean.csv)
Saved cleaned NIWIS_WasteWaterQuality_16-Jul-2026  ((143, 5)) to Outputs/CleanData/NIWIS_WasteWaterQuality_16-Jul-2026 _clean.csv)
Saved cleaned Potable Systems_ 2026_04_21 ((61, 11)) to Outputs/CleanData/Potable Systems_ 2026_04_21_clean.csv)
Saved cleaned Surface Water Storage (2014-2020) ((374, 9)) to Outputs/CleanData/Surface Water Storage (2014-2020)_clean.csv)
Saved cleaned TimeSeries Population_ 2026_04_21 ((11, 35)) to Outputs/CleanData/TimeSeries Population_ 2026_04_21_clean.csv)
Saved cleaned Vulnerability Score Per leg_ 2026_04_21 ((11, 9)) to Outputs/CleanData/Vulnerability Score Per leg_ 2026_04_21_clean.csv)
Saved cleaned Operational_ 2026_04_21 ((11, 8)) to Outputs/CleanData/Operational_ 2026_04_21_clean.csv)
Saved cleaned Piped Water Households _ 2026_04_21 ((10, 12)) to Outputs/CleanData/Piped Water Households _ 2026_04_21_clean.csv)
Saved cleane